# **Cellpose in Google Colab — Fine-tuning & Inference**

In this notebook, we fine-tune the Cellpose *cyto3* model on the **tricho** dataset.

**Cellpose version:** `3.1.1.1`

Training images and their corresponding masks (in `_seg.npy` or `_masks.tif` format) are required to run the fine-tuning. Images should ideally be uploaded to Google Drive before starting this session.

After training, the fine-tuned model is compared side-by-side against the original *cyto3* to evaluate the improvement. The fine-tuned model should be better adapted to the specific cell type.






---

### 📁 How to load your data into Google Colab

You have two easy options:

#### Option A — Google Drive (recommended for large datasets)
1. Upload your images and masks to a folder in your Google Drive, e.g.:
   ```
   MyDrive/train_example/
        img001.tif
        img001_seg.npy    ← Cellpose GUI output, same base name + _seg suffix
        img002.tif
        img002_seg.npy
   ```
2. Run **Section 1** below to mount Drive and set the path.

#### Option B — Direct upload (small datasets / quick tests)
- Use the upload button in Files navigation or code below. Files uploaded this way land in the current working directory, which in Colab is `/content/` by default.
    ```python
    from google.colab import files
    import os

    uploaded = files.upload()

    print(os.listdir('/content'))
    ```


---

## **0 — Install Cellpose 3.1.1.1**

In [ ]:
!pip install -q cellpose==3.1.1.1

import cellpose, torch
print(f"Cellpose version : {cellpose.version}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")

---

## **1 — Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ✏️  Edit this path to match your Google Drive folder
GDRIVE_DATA_ROOT = '/content/drive/MyDrive/training_example'

---

## **2 — Load & Preview Training Data**

This section pairs image files with their `_seg.npy` masks.  
It also supports plain `_masks.tif` / `_masks.png` files as a fallback.


*NOTE:* `_seg.npy` files are produced by the Cellpose GUI.

They are dictionaries containing a `masks` array (integer labels: 0=background, 1,2,3…=objects).  

In [ ]:
#@markdown ## Detecting training data

#@markdown Run this cell to read training data (image - mask pairs).

#@markdown A preview of the first image/mask pair is shown to verify the data loaded correctly.
import os
import numpy as np
import matplotlib.pyplot as plt
from cellpose import io

IMAGE_EXTENSIONS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')

def load_image_mask_pairs(train_dir):
    images, masks, names = [], [], []

    img_files = sorted([
        f for f in os.listdir(train_dir)
        if f.lower().endswith(IMAGE_EXTENSIONS)
        and '_masks' not in f
    ])

    for fname in img_files:
        base = os.path.splitext(fname)[0]
        img_path = os.path.join(train_dir, fname)

        # --- try _seg.npy first ---
        seg_path = os.path.join(train_dir, base + '_seg.npy')
        if os.path.exists(seg_path):
            seg = np.load(seg_path, allow_pickle=True).item()
            mask = seg['masks'].astype(np.int32)
            images.append(io.imread(img_path))
            masks.append(mask)
            names.append(base)
            print(f"  ✅  {fname}  +  {base}_seg.npy  →  {mask.max()} cells")
            continue

        # --- fallback: _masks.tif / _masks.png ---
        for ext in IMAGE_EXTENSIONS:
            mask_path = os.path.join(train_dir, base + '_masks' + ext)
            if os.path.exists(mask_path):
                mask = io.imread(mask_path).astype(np.int32)
                images.append(io.imread(img_path))
                masks.append(mask)
                names.append(base)
                print(f"  ✅  {fname}  +  {base}_masks{ext}  →  {mask.max()} cells")
                break
        else:
            print(f"  ⚠️  {fname}  — no matching mask found, skipping")

    return images, masks, names


print(f"Loading from: {GDRIVE_DATA_ROOT}")
images_train, masks_train, names_train = load_image_mask_pairs(GDRIVE_DATA_ROOT)
print(f"\nLoaded {len(images_train)} image/mask pair(s)")

# --- Preview first pair ---
if images_train:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    img0 = images_train[0]
    axes[0].imshow(img0, cmap='gray')
    axes[0].set_title(f'Example image: {names_train[0]}')
    axes[0].axis('off')
    axes[1].imshow(masks_train[0], cmap='tab20b')
    axes[1].set_title(f'Mask: {masks_train[0].max()} cells')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## **3 — Fine-tune cyto3 model**

Set configuration from model training.

In [ ]:
#@markdown ### Training Configuration

#@markdown ##### Pretrained model to fine-tune
pretrained_model = 'cyto3' #@param ["cyto3", "cyto2", "cyto", "nuclei"]

#@markdown ##### Output model name
MODEL_NAME = 'cyto3_finetuned' #@param {type:"string"}

#@markdown ##### Mean cell diameter (pixels)
DIAMETER = 60 #@param {type:"integer"}

#@markdown ##### Channel setup
CHANNELS_STR = '0,0' #@param ["0,0","0,1","0,2","0,3","1,0","1,1","1,2","1,3","2,0","2,1","2,2","2,3","3,0","3,1","3,2","3,3"]
CHANNELS = [int(c) for c in CHANNELS_STR.split(',')]

#@markdown ##### Epochs
N_EPOCHS = 200 #@param {type:"integer"}

#@markdown ##### Learning rate
LEARNING_RATE = 0.1 #@param {type:"number"}

#@markdown ##### Weight decay
WEIGHT_DECAY = 0.0001 #@param {type:"number"}

#@markdown ##### Batch size
BATCH_SIZE = 8 #@param {type:"integer"}

print("── Setup Configuration ──────────────────────")
print(f"  Pretrained model : {pretrained_model}")
print(f"  Output model name: {MODEL_NAME}")
print(f"  Diameter         : {DIAMETER} px")
print(f"  Channels         : {CHANNELS}")
print(f"  Epochs           : {N_EPOCHS}")
print(f"  Learning rate    : {LEARNING_RATE}")
print(f"  Weight decay     : {WEIGHT_DECAY}")
print(f"  Batch size       : {BATCH_SIZE}")
print("────────────────────────────────────────────────")

In [ ]:
#@markdown ### Training

#@markdown Run this cell to fine-tune the model with the parameters defined above.

from cellpose import models, train

use_gpu = torch.cuda.is_available()
print(f"Training on {'GPU 🚀' if use_gpu else 'CPU'}")

model = models.CellposeModel(gpu=use_gpu, model_type=pretrained_model)

new_model = train.train_seg(
    model.net,
    train_data=images_train,
    train_labels=masks_train,
    train_files=None,
    channels=CHANNELS,
    normalize=True,
    save_path=GDRIVE_DATA_ROOT,
    save_every=25,
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    min_train_masks=1
)

new_model_path = os.path.join(GDRIVE_DATA_ROOT, 'models', MODEL_NAME)
print(f"\n✅  Fine-tuned model saved to: {new_model_path}")

## **4 — Segmentation with the new model**

In [ ]:
#@markdown ## Inference

#@markdown Run this cell to segment images using the fine-tuned model.
#@markdown All images found in the data directory are loaded and segmented automatically.

# ── Settings ────────────────────────────────────────────────
SEGMENT_MODEL_PATH = new_model_path
SEGMENT_DIR = GDRIVE_DATA_ROOT
CHANNELS           = [0, 0]
FLOW_THRESHOLD     = 0.4
CELLPROB_THRESHOLD = 0.0
DIAMETER           = 60
# ────────────────────────────────────────────────────────────


test_files = [
    os.path.join(SEGMENT_DIR, f)
    for f in sorted(os.listdir(SEGMENT_DIR))
    if not f.endswith('_seg.npy') and '_masks' not in f
    and f.lower().endswith(IMAGE_EXTENSIONS)
]
print(f"Found {len(test_files)} test image(s): {[os.path.basename(p) for p in test_files]}")

test_images = [io.imread(p) for p in test_files]

seg_model = models.CellposeModel(gpu=use_gpu, pretrained_model=SEGMENT_MODEL_PATH)

masks_pred, flows, styles = seg_model.eval(
    test_images,
    diameter=DIAMETER,
    channels=CHANNELS,
    flow_threshold=FLOW_THRESHOLD,
    cellprob_threshold=CELLPROB_THRESHOLD,
    batch_size=BATCH_SIZE
)

for i, mask in enumerate(masks_pred):
    print(f"{os.path.basename(test_files[i])}: {mask.max()} cells detected")

In [ ]:
#@markdown ## Show segmentation results
#@markdown Run this cell to display results.

import matplotlib.pyplot as plt
from cellpose import plot as cpplot

n_show = min(len(test_images), 5)
fig, axes = plt.subplots(n_show, 3, figsize=(10, 1 * n_show))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    img  = test_images[i]
    mask = masks_pred[i]

    disp = img if img.ndim == 2 else img[..., 0]
    axes[i][0].imshow(disp, cmap='gray')
    axes[i][0].set_title(f'Image {[os.path.basename(test_files[i])]}')
    axes[i][0].axis('off')

    axes[i][1].imshow(mask, cmap='tab20b')
    axes[i][1].set_title(f'Mask — {mask.max()} cells')
    axes[i][1].axis('off')

    img_rgb = np.stack([disp]*3, axis=-1) if img.ndim == 2 else img[..., :3]
    axes[i][2].imshow(cpplot.mask_overlay(img_rgb, mask))
    axes[i][2].set_title('Overlay')
    axes[i][2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#@markdown ## Save predicted masks (as TIF)

#@markdown Run this cell to save predicted masks to the connected Google Drive.
from cellpose import io as cpio
import os, numpy as np

SAVE_MASK_DIR = os.path.join(SEGMENT_DIR, 'predictions')
os.makedirs(SAVE_MASK_DIR, exist_ok=True)

for fpath, mask in zip(test_files, masks_pred):
    base = os.path.splitext(os.path.basename(fpath))[0]
    out  = os.path.join(SAVE_MASK_DIR, f'{base}_cp_masks.tif')
    cpio.imsave(out, mask.astype(np.uint16))   # uint16 supports >255 cells
    print(f"Saved → {out}")

print(f"\n✅  All masks saved to {SAVE_MASK_DIR}")

## **5 — Compare with original cyto3**

In [ ]:
#@markdown ## Model performance comparison

#@markdown Run this cell to segment the same images with the original *cyto3* model and compare results side-by-side.

baseline = models.CellposeModel(gpu=use_gpu, model_type='cyto3')
masks_baseline, _, _ = baseline.eval(
    test_images, diameter=DIAMETER, channels=CHANNELS,
    flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD
)

n_show = min(len(test_images), 5)
fig, axes = plt.subplots(n_show, 3, figsize=(10, 1 * n_show))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    disp = test_images[i] if test_images[i].ndim == 2 else test_images[i][..., 0]

    axes[i][0].imshow(disp, cmap='gray')
    axes[i][0].set_title(f'Image {[os.path.basename(test_files[i])]}')

    axes[i][1].imshow(masks_baseline[i], cmap='tab20b')
    axes[i][1].set_title(f'cyto3 — {masks_baseline[i].max()} cells')

    axes[i][2].imshow(masks_pred[i], cmap='tab20b')
    axes[i][2].set_title(f'Fine-tuned — {masks_pred[i].max()} cells')

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()